# 🗳️ VOTING ENSEMBLE (OY BİRLİĞİ TOPLULUK MODELİ)

> **🎯 AMAÇ:** Birden fazla modeli birleştirerek daha güçlü ve kararlı tahmin  
> **📥 GİRDİ:** X_train, X_test, y_train, y_test (Bellekten - Ölçeklenmiş)  
> **📤 ÇIKTI:** Accuracy, Confusion Matrix, Model Karşılaştırma Grafiği  

### ⏱️ NE ZAMAN KULLANILIR?
- Tek başına iyi olan modelleri birleştirip daha iyi sonuç almak için
- Varyansı ve bias'ı azaltmak için
- Kaggle/yarışma son aşamalarında puanı artırmak için

### 🔧 PARAMETRELER
- `voting='hard'` → Çoğunluk oyu
- `voting='soft'` → Olasılık ortalaması (genelde daha iyi)
- `weights=[...]` → Her modele farklı ağırlık

---

### ⚠️ UYARI
```python
%run Veri_On_Isleme.py
```
(X_train, X_test, y_train, y_test bellekte olmalı)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

print('=' * 50)
print('🗳️ VOTING ENSEMBLE BAŞLATILIYOR')
print('=' * 50)

In [ ]:
estimators = [
    ('rf',  RandomForestClassifier(n_estimators=100, random_state=42)),
    ('lr',  LogisticRegression(max_iter=1000, random_state=42)),
    ('svm', SVC(probability=True, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('dt',  DecisionTreeClassifier(random_state=42))
]

voting_soft = VotingClassifier(estimators=estimators, voting='soft', weights=[2,1,1,1,1])
voting_hard = VotingClassifier(estimators=estimators, voting='hard')

voting_soft.fit(X_train, y_train.values.ravel())
voting_hard.fit(X_train, y_train.values.ravel())
print('✅ Her iki model eğitildi.')

In [ ]:
y_pred_soft = voting_soft.predict(X_test)
y_pred_hard = voting_hard.predict(X_test)
acc_soft = accuracy_score(y_test, y_pred_soft)
acc_hard = accuracy_score(y_test, y_pred_hard)

results = {}
for name, model in estimators:
    model.fit(X_train, y_train.values.ravel())
    results[name] = accuracy_score(y_test, model.predict(X_test))
results['Soft Voting'] = acc_soft
results['Hard Voting'] = acc_hard

for k, v in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f'  {k:<15}: %{v*100:.2f}')
print(f'\n🏆 SOFT: %{acc_soft*100:.2f} | HARD: %{acc_hard*100:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#2196F3' if 'Voting' not in k else '#FF5722' for k in results.keys()]
bars = axes[0].barh(list(results.keys()), [v*100 for v in results.values()], color=colors)
axes[0].set_title('Model Karşılaştırması')
cm_soft = confusion_matrix(y_test, y_pred_soft)
sns.heatmap(cm_soft, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title(f'Soft Voting (Acc: %{acc_soft*100:.1f})')
cm_hard = confusion_matrix(y_test, y_pred_hard)
sns.heatmap(cm_hard, annot=True, fmt='d', cmap='Oranges', ax=axes[2])
axes[2].set_title(f'Hard Voting (Acc: %{acc_hard*100:.1f})')
plt.tight_layout(); plt.show()
print('\n✅ İşlem Tamamlandı.')